# Residential Module — Step-by-step Tutorial

This notebook walks through the residential sector pipeline interactively.

In [ ]:
import sys, pathlib

repo_root = pathlib.Path().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Python:", sys.version)
print("Repo root:", repo_root)

## 2 — End-use structure

The residential module models five end-uses:
space heating, space cooling, water heating, cooking, appliances.

In [ ]:
import pandas as pd
from residential.build_energy_intensities import build_energy_intensities

intensities = build_energy_intensities()
pd.DataFrame.from_dict(intensities, orient='index', columns=['kWh / unit'])

## 3 — Macro indicators for a single country

Population, GDP/capita, floor area per person, and persons per dwelling
are the activity drivers.

In [ ]:
from residential.build_macro_indicators import build_macro_indicators

COUNTRY = "Morocco"
SCENARIOS = ["Current Policies", "Progressive", "Ambitious Net Zero"]
SSP_ORDER  = ["SSP2", "SSP2", "SSP1"]
YEARS = list(range(2018, 2051))

macro = build_macro_indicators(COUNTRY, SCENARIOS, SSP_ORDER, YEARS)
macro.head(10)

## 4 — Run the full residential pipeline

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(repo_root / "residential" / "build_residential_config.py")],
    capture_output=True, text=True, cwd=repo_root,
)
print(result.stdout[-2000:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 5 — Inspect combined parameter file

In [ ]:
params_path = repo_root / "residential" / "output_data" / "parameters_all_countries.csv"
if params_path.exists():
    params = pd.read_csv(params_path)
    print(params.shape)
    params[params["country"] == COUNTRY].head(15)
else:
    print(f"Not found: {params_path}")

## Next steps

- Change `COUNTRY` in cell 3 to explore a different country.
- Modify scenario assumptions in `residential/build_residential_config.py`.
- See the [Residential Methodology](../methodology/residential.md) page.